In [ ]:
%run ./1_config


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

class GoldAnalyticsBuilder:
    def __init__(self):
        self.c = conf
        self.ev = spark.table(self.c.table_fqn(self.c.silver_events_enriched))

    def build_geo_daily(self):
        df = (self.ev.groupBy(F.to_date("event_ts").alias("date"), "country_code")
              .agg(F.count("*").alias("event_count"),
                   F.countDistinct("session_id").alias("unique_sessions"),
                   F.sum(F.coalesce("spend", F.lit(0.0))).alias("revenue")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_geo_traffic_daily))

    def build_fraud(self):
        w = Window.partitionBy("session_id")
        df = (self.ev.withColumn("event_count_1h", F.count("*").over(w))
              .select("session_id","ip","country_code","is_vpn",
                      F.when(F.col("is_vpn"), 1).otherwise(0).alias("vpn_risk_score"),
                      "event_count_1h")
              .dropDuplicates(["session_id"])
              .withColumn("flag_suspicious", (F.col("is_vpn") == True) & (F.col("event_count_1h") >= 10)))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_fraud_signals))

    def build_customer(self):
        df = (self.ev.groupBy("user_id")
              .agg(F.first("country_code", ignorenulls=True).alias("primary_country"),
                   F.first("timezone_id", ignorenulls=True).alias("timezone"),
                   F.sum(F.coalesce("spend", F.lit(0.0))).alias("total_spend"),
                   F.countDistinct("session_id").alias("session_count"),
                   F.max("event_ts").alias("last_seen_at")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_customer_features))

    def run(self):
        self.build_geo_daily()
        self.build_fraud()
        self.build_customer()
        print("✅ Gold tables built")

    def validate(self):
        for t in [self.c.gold_geo_traffic_daily, self.c.gold_fraud_signals, self.c.gold_customer_features]:
            n = spark.table(self.c.table_fqn(t)).count()
            print(f"🔎 {t}: {n} rows")

gb = GoldAnalyticsBuilder()
gb.run()
gb.validate()

